<a href="https://colab.research.google.com/github/kasrasa/ViT-VLM-experiments/blob/ViT-Experiments/ViT_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers torch

In [ ]:
!pip install -q timm

In [ ]:
!pip install -q evaluate

In [ ]:
import os

# Set to True to always fine-tune, False to load existing checkpoints if available
FORCE_FINE_TUNING = False

In [ ]:
import timm
from transformers import AutoImageProcessor, AutoModelForImageClassification

# Using a smaller ViT model for reduced RAM usage
model_id = "vit_base_patch16_224"

model = timm.create_model(model_id, pretrained=True)
model.eval()

config = timm.data.resolve_model_data_config(model)
print(config)
vit_transform = timm.data.create_transform(**config)


# Load an AutoModelForImageClassification from Hugging Face Transformers to get ImageNet-1k id2label mapping
# This will be used for both ViT and ResNet predictions for consistency.
id2label_model = AutoModelForImageClassification.from_pretrained("google/vit-base-patch16-224")
id2label_mapping = id2label_model.config.id2label

print(f"Loaded ViT model: {model_id}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from datasets import load_dataset

# Load the full training split first
food_full_train = load_dataset("ethz/food101", split="train")

# Shuffle the full training split and select the first 5000 examples
# This ensures a more diverse set of classes in the subset
food = food_full_train.shuffle(seed=42).select(range(5000))

# Now perform the train-test split on this diverse subset
food = food.train_test_split(test_size=0.2, shuffle=True, seed=42)

In [ ]:
labels = food["train"].features["label"].names
label2id, id2label = dict(), dict()
for i, label in enumerate(labels):
    label2id[label] = str(i)
    id2label[str(i)] = label

In [ ]:
from transformers import AutoImageProcessor

checkpoint = "google/vit-base-patch16-224-in21k"
image_processor = AutoImageProcessor.from_pretrained(checkpoint)

In [ ]:
from torchvision.transforms import RandomResizedCrop, Compose, Normalize, ToTensor, CenterCrop

# Get normalization values and size from the image_processor for ViT
normalize_vit = Normalize(mean=image_processor.image_mean, std=image_processor.image_std)
size_vit = (
    image_processor.size["shortest_edge"]
    if "shortest_edge" in image_processor.size
    else (image_processor.size["height"], image_processor.size["width"])
)

# Define separate transforms for training and validation for ViT
vit_transforms_train = Compose([RandomResizedCrop(size_vit), ToTensor(), normalize_vit])
vit_transforms_val = Compose([CenterCrop(size_vit), ToTensor(), normalize_vit])


In [ ]:
def vit_transforms_function(examples):
    # Apply appropriate transforms for training vs. evaluation
    if "image" in examples:
        examples["pixel_values"] = [vit_transforms_train(img.convert("RGB")) for img in examples["image"]]
    return examples

def vit_val_transforms_function(examples):
    if "image" in examples:
        examples["pixel_values"] = [vit_transforms_val(img.convert("RGB")) for img in examples["image"]]
    return examples

# Re-apply transforms to the dataset for ViT, using separate train/test functions
# Note: This will re-shuffle and re-split the 'food' dataset, so subsequent ViT training will be on new splits
food_vit = food_full_train.shuffle(seed=42).select(range(5000))
food_vit = food_vit.train_test_split(test_size=0.2, shuffle=True, seed=42)
food_vit["train"] = food_vit["train"].with_transform(vit_transforms_function)
food_vit["test"] = food_vit["test"].with_transform(vit_val_transforms_function)

print("ViT-specific image transforms updated for separate training (RandomResizedCrop) and validation (CenterCrop).")


In [ ]:
from transformers import DefaultDataCollator

data_collator = DefaultDataCollator()

In [ ]:
from transformers import AutoModelForImageClassification, TrainingArguments, Trainer

model = AutoModelForImageClassification.from_pretrained(
    checkpoint,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id,
)

## Update `compute_metrics` Function

### Subtask:
Modify the `compute_metrics` function to include the F1-score in addition to accuracy. This will provide a more comprehensive evaluation metric, especially for imbalanced datasets.

In [ ]:
import evaluate
import numpy as np

# Load the F1 metric
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    # eval_pred is a PredictionOutput object
    predictions_logits = eval_pred.predictions
    labels = eval_pred.label_ids

    predictions = np.argmax(predictions_logits, axis=1)

    # Compute accuracy
    accuracy_result = accuracy.compute(predictions=predictions, references=labels)

    # Compute F1-score. Use 'weighted' average for multi-class classification.
    f1_result = f1_metric.compute(predictions=predictions, references=labels, average="weighted")

    # Compute Top-5 Accuracy
    # For top-k accuracy, we need to sort the logits and check if the true label is among the top-k
    k = 5
    top_k_predictions = np.argsort(predictions_logits, axis=1)[:, -k:] # Get indices of top k predictions
    top_k_accuracy = np.mean([labels[i] in top_k_predictions[i] for i in range(len(labels))])

    # Combine results
    return {
        "accuracy": accuracy_result["accuracy"],
        "f1": f1_result["f1"],
        "top_5_accuracy": top_k_accuracy
    }

In [ ]:
from transformers.trainer_utils import get_last_checkpoint
import os
import torch
from safetensors.torch import load_file # Import for .safetensors models

def conditional_train_model(model_instance, trainer_instance, training_args_instance, model_name_for_log):
    checkpoint_dir = training_args_instance.output_dir
    os.makedirs(checkpoint_dir, exist_ok=True)

    # Check for a 'final' model saved directly in the output_dir
    final_model_root_path = os.path.join(checkpoint_dir, "pytorch_model.bin")
    final_model_root_safetensors_path = os.path.join(checkpoint_dir, "model.safetensors")

    # Try to find the latest checkpoint subdirectory (e.g., path/to/output_dir/checkpoint-XXX)
    latest_checkpoint_subdir_path = None
    if os.path.isdir(checkpoint_dir):
        latest_checkpoint_subdir_path = get_last_checkpoint(checkpoint_dir)

    model_exists = False
    # Condition 1: Final model (pytorch_model.bin or model.safetensors) exists at the root of output_dir
    if os.path.exists(final_model_root_path) or os.path.exists(final_model_root_safetensors_path):
        model_exists = True
        print(f"Final model found at root for {model_name_for_log} at {checkpoint_dir}.")
    # Condition 2: No final model at root, but a checkpoint subdirectory exists and contains a model file
    elif latest_checkpoint_subdir_path and \
        (os.path.exists(os.path.join(latest_checkpoint_subdir_path, "pytorch_model.bin")) or \
         os.path.exists(os.path.join(latest_checkpoint_subdir_path, "model.safetensors"))):
        model_exists = True
        print(f"Model checkpoint found in subdirectory for {model_name_for_log} at {latest_checkpoint_subdir_path}.")
        # If we are skipping training based on a subdirectory checkpoint, we need to explicitly
        # load that checkpoint into the model_instance so that subsequent evaluations use the correct weights.
        try:
            if os.path.exists(os.path.join(latest_checkpoint_subdir_path, "pytorch_model.bin")):
                model_instance.load_state_dict(torch.load(os.path.join(latest_checkpoint_subdir_path, "pytorch_model.bin")))
            elif os.path.exists(os.path.join(latest_checkpoint_subdir_path, "model.safetensors")):
                model_instance.load_state_dict(load_file(os.path.join(latest_checkpoint_subdir_path, "model.safetensors")))
            print(f"Model {model_name_for_log} loaded from {latest_checkpoint_subdir_path}.")
        except Exception as e:
            print(f"Warning: Could not load model {model_name_for_log} from checkpoint {latest_checkpoint_subdir_path}: {e}")
            model_exists = False # If loading fails, treat as if no model exists

    train_runtime = 0.0 # Default to 0 if no training occurs

    if model_exists and not FORCE_FINE_TUNING:
        print(f"Skipping training for {model_name_for_log} as a trained model was found and FORCE_FINE_TUNING is False.")
        # When skipping, the Trainer will use the already loaded model (if manually loaded above)
        # or it will internally load the best one for evaluation/prediction if load_best_model_at_end=True.
        # So we just return.
    else:
        # If FORCE_FINE_TUNING is True, or no model was found, or loading failed
        if FORCE_FINE_TUNING:
            print(f"FORCE_FINE_TUNING is True for {model_name_for_log}. Starting fine-tuning from scratch.")
            train_result = trainer_instance.train(resume_from_checkpoint=None) # Start fresh
        elif latest_checkpoint_subdir_path: # Only resume if a valid checkpoint was found and not force fine-tuning
            print(f"No final model found at root, but existing checkpoint found for {model_name_for_log} at {latest_checkpoint_subdir_path}. Resuming training.")
            train_result = trainer_instance.train(resume_from_checkpoint=latest_checkpoint_subdir_path) # Resume
        else:
            print(f"No existing model or checkpoint found for {model_name_for_log}. Starting fine-tuning.")
            train_result = trainer_instance.train() # Start fresh

        train_runtime = train_result.metrics["train_runtime"]
        print(f"Fine-tuning for {model_name_for_log} complete.")

    return train_runtime

### Freezing Backbone Layers

To fine-tune only the classification head, we need to freeze the parameters of the model's feature extractor (the backbone). This means we'll set `requires_grad=False` for all parameters except those belonging to the `model.head` module.

In [ ]:
print(model)

# Freeze all layers first
for param in model.parameters():
    param.requires_grad = False

# Unfreeze the classification head (typically named 'head' in timm models)
# You might need to inspect the model architecture (e.g., print(model)) if it's named differently
for param in model.classifier.parameters():
    param.requires_grad = True

print("Model parameters frozen. Only classification head will be fine-tuned.")

In [ ]:
import evaluate
accuracy = evaluate.load("accuracy")

In [ ]:
training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/ViTExperiments/my_awesome_food_model_head_only", # Saving to Google Drive
    remove_unused_columns=False,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-4,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=4,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    warmup_steps=0.1,
    logging_steps=10,
    run_name="food101",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    push_to_hub=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=food["train"],
    eval_dataset=food["test"],
    processing_class=image_processor,
    compute_metrics=compute_metrics,
)

print("Starting ViT Head-Only fine-tuning...")
# Use the conditional training function
vit_head_only_train_time = conditional_train_model(model, trainer, training_args, "ViT Head-Only Fine-tune")
print("ViT Head-Only fine-tuning complete.")

## Fine-tuning the Whole Model (All Layers Unfrozen)

As requested, this section will fine-tune the entire Vision Transformer model (all layers, not just the classification head) on the Food101 dataset for 10 epochs.

In [ ]:
# Re-initialize the model to ensure no layers are frozen from previous steps
model_full_finetune = AutoModelForImageClassification.from_pretrained(
    checkpoint,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id,
)

# Explicitly ensure all parameters require gradients for full fine-tuning
for param in model_full_finetune.parameters():
    param.requires_grad = True

print("Model re-initialized. All parameters are unfrozen and will be trained.")
print(model_full_finetune)

In [ ]:
training_args_full = TrainingArguments(
    output_dir="/content/drive/MyDrive/ViTExperiments/my_awesome_food_model_full_finetune", # Saving to Google Drive
    remove_unused_columns=False,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=4,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    warmup_steps=0.1,
    logging_steps=10,
    run_name="food101_full_finetune",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    push_to_hub=False,
)

trainer_full = Trainer(
    model=model_full_finetune,
    args=training_args_full,
    data_collator=data_collator,
    train_dataset=food["train"],
    eval_dataset=food["test"],
    processing_class=image_processor,
    compute_metrics=compute_metrics,
)

print("Starting full model fine-tuning...")
# Use the conditional training function
vit_full_train_time = conditional_train_model(model_full_finetune, trainer_full, training_args_full, "ViT Full Fine-tune")
print("Full model fine-tuning complete.")

## LoRA Fine-tuning

This section demonstrates fine-tuning the Vision Transformer model using Low-Rank Adaptation (LoRA) to reduce computational cost and memory footprint during training.

In [ ]:
!pip install -q peft
!pip install --upgrade -q torchao

In [ ]:
from peft import LoraConfig, get_peft_model

# Define LoRA configuration
lora_config_dict = {
    "r": 16,  # LoRA attention dimension
    "lora_alpha": 32,  # Alpha parameter for LoRA scaling
    "target_modules": ["query", "value"], # Target modules for LoRA. For ViT, 'query' and 'value' are common.
    "lora_dropout": 0.1,  # Dropout probability for LoRA layers
    "bias": "none",  # Bias type for LoRA layers
    "task_type": "CAUSAL_LM" # Task type. Set to 'CAUSAL_LM' for now, will adjust if needed.
}

# Re-initialize the base model
model_lora_base = AutoModelForImageClassification.from_pretrained(
    checkpoint,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id,
)

# Adjust task type if necessary for image classification
# The task type 'CAUSAL_LM' is often used for text, for image classification, a different task_type might be more appropriate
# For image classification, there isn't a direct PEFT task_type, but we'll adapt.
# Let's set it to 'SEQ_CLS' (sequence classification) which is a common fallback for classification tasks if no specific image task type exists.
lora_config_dict['task_type'] = 'SEQ_CLS'
lora_config = LoraConfig(**lora_config_dict)

# Wrap the base model with LoRA
model_lora = get_peft_model(model_lora_base, lora_config)

print("LoRA model created:")
model_lora.print_trainable_parameters()
print(model_lora)

In [ ]:
# Set up TrainingArguments for LoRA fine-tuning
training_args_lora = TrainingArguments(
    output_dir="/content/drive/MyDrive/ViTExperiments/my_awesome_food_model_lora", # Saving to Google Drive
    remove_unused_columns=False,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-4,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=4,
    per_device_eval_batch_size=16,
    num_train_epochs=5, # Start with a few epochs for LoRA
    warmup_steps=0.1,
    logging_steps=10,
    run_name="food101_lora_finetune",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    push_to_hub=False,
)

trainer_lora = Trainer(
    model=model_lora,
    args=training_args_lora,
    data_collator=data_collator,
    train_dataset=food["train"],
    eval_dataset=food["test"],
    processing_class=image_processor,
    compute_metrics=compute_metrics,
)

print("Starting LoRA fine-tuning...")
# Use the conditional training function
vit_lora_train_time = conditional_train_model(model_lora, trainer_lora, training_args_lora, "ViT LoRA Fine-tune")
print("LoRA fine-tuning complete.")

# Task
The goal is to compare different fine-tuning strategies (head-only, full model, and LoRA) for a Vision Transformer (ViT) model on the Food101 dataset. The comparison will be based on performance metrics (accuracy and F1-score) and the number of trainable parameters. Additionally, the models will be evaluated on a separate CIFAR-10 dataset to assess their generalization capabilities.

## Install Additional Libraries

### Subtask:
Install necessary libraries such as `scikit-learn` for F1 score and confusion matrix, `matplotlib` and `seaborn` for plotting, and `accelerate` for optimized training.


In [ ]:
!pip install -q scikit-learn

In [ ]:
!pip install -q matplotlib seaborn accelerate

## Define Parameter Counting Utility

### Subtask:
Create a Python function to count and display the number of trainable parameters in a given PyTorch model. This function will be reused at various stages to track model complexity.


**Reasoning**:
The subtask requires defining a function to count trainable parameters. This code block implements that function by iterating through model parameters and summing up the elements of those that require gradients, then printing the result in millions.



In [ ]:
def count_parameters(model):
    """
    Counts and displays the number of trainable parameters in a PyTorch model.
    """
    num_params_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Number of trainable parameters: {num_params_trainable:,} ({num_params_trainable / 1e6:.2f} million)")
    print(f"Total parameters: {total_params:,} ({total_params / 1e6:.2f} million)")


## Define Reusable Evaluation and Plotting Function

### Subtask:
Create a function that takes a trained model, a dataset, and label mappings as input, computes predictions, calculates accuracy and F1-score, and generates a confusion matrix. This function will be used to evaluate all fine-tuned models on both Food101 and CIFAR-10 datasets.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report # Added classification_report
import torch
from tqdm.auto import tqdm
import numpy as np # Ensure numpy is imported for argmax and nan_to_num
import pandas as pd # Added pandas for DataFrame operations

def evaluate_and_plot(model, trainer, dataset, id2label_mapping, dataset_name, num_labels, model_display_name, training_time=None, normalize_cm=False, device='cuda'):
    print(f"\n--- Evaluating on {dataset_name} ---")

    # Make predictions
    predictions_output = trainer.predict(dataset) # Renamed to avoid confusion with predicted_labels
    logits = predictions_output.predictions
    labels = predictions_output.label_ids

    # Get predicted labels
    predicted_labels = np.argmax(logits, axis=1)

    # Print unique labels to verify distribution
    print(f"Unique true labels in {dataset_name}: {np.unique(labels)}")
    print(f"Unique predicted labels in {dataset_name}: {np.unique(predicted_labels)}")

    # Compute metrics using the shared compute_metrics function
    metrics = compute_metrics(predictions_output) # Use predictions_output here
    print(f"Accuracy on {dataset_name}: {metrics['accuracy']:.4f}")
    print(f"F1-score (weighted) on {dataset_name}: {metrics['f1']:.4f}")
    print(f"Top-5 Accuracy on {dataset_name}: {metrics['top_5_accuracy']:.4f}")

    if training_time is not None:
        print(f"Training time for {model_display_name}: {training_time:.2f} seconds")

    # Generate Confusion Matrix (Full)
    cm = confusion_matrix(labels, predicted_labels)

    if normalize_cm:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        cm = np.nan_to_num(cm) # Handle cases where a row sums to zero (no true instances of a class)

    # Plot Full Confusion Matrix
    plt.figure(figsize=(12, 10))

    # Conditional plotting for readability with many classes
    if num_labels > 20:
        sns.heatmap(cm, cmap='Blues',
                    xticklabels=False, yticklabels=False, # Disable tick labels for clarity
                    cbar=True) # Keep color bar
        plt.title(f'Confusion Matrix for {model_display_name} on {dataset_name} (Labels Omitted)')
    else:
        sns.heatmap(cm, annot=True, fmt='.2f' if normalize_cm else 'g', cmap='Blues',
                    xticklabels=[id2label_mapping[str(i)] for i in range(num_labels)], # Use id2label_mapping for labels
                    yticklabels=[id2label_mapping[str(i)] for i in range(num_labels)]) # Use id2label_mapping for labels
        plt.title(f'Confusion Matrix for {model_display_name} on {dataset_name}')

    plt.xlabel('Predicted labels')
    plt.ylabel('True labels')
    plt.tight_layout() # Adjust layout to prevent labels from being cut off
    plt.show()

    # --- New functionality for misclassified classes and per-class metrics ---
    print(f"\n--- Per-Class Metrics and Top Misclassified Classes for {model_display_name} on {dataset_name} ---")

    # Generate classification report
    # Ensure labels passed to classification_report are consistent with id2label_mapping for keys
    target_names = [id2label_mapping[str(i)] for i in sorted([int(k) for k in id2label_mapping.keys()])]
    report = classification_report(labels, predicted_labels, output_dict=True, zero_division=0, target_names=target_names)

    # Convert report to DataFrame for easy sorting and display
    report_df = pd.DataFrame(report).transpose()
    # Filter out 'accuracy', 'macro avg', 'weighted avg' rows if present and ensure f1-score is numeric
    class_metrics_df = report_df[report_df.index.isin(target_names)].copy()
    class_metrics_df['f1-score'] = pd.to_numeric(class_metrics_df['f1-score'], errors='coerce')

    # Sort by f1-score to find the most misclassified classes (lowest f1-score)
    misclassified_classes = class_metrics_df.sort_values(by='f1-score', ascending=True).head(10) # Top 10

    print("\nTop 10 Misclassified Classes (by F1-score):")
    display(misclassified_classes[['precision', 'recall', 'f1-score', 'support']].round(2))

    # Plot refined confusion matrix for these top misclassified classes
    if not misclassified_classes.empty:
        # Get the original integer labels for these classes
        # Map class names back to their integer IDs for confusion matrix creation
        label2id_inverse = {v: int(k) for k, v in id2label_mapping.items()}
        misclassified_label_indices = [label2id_inverse[class_name] for class_name in misclassified_classes.index.tolist()]
        misclassified_class_names = misclassified_classes.index.tolist()

        # Filter true and predicted labels to include only these classes
        # Ensure we filter both based on true labels being in misclassified_label_indices
        mask = np.isin(labels, misclassified_label_indices)
        filtered_labels = labels[mask]
        filtered_predicted_labels = predicted_labels[mask]

        # If after filtering, there are no samples for these classes in the current batch (unlikely for test set but good to check)
        if len(filtered_labels) > 0:
            # Create a confusion matrix specifically for these misclassified classes
            cm_refined = confusion_matrix(filtered_labels, filtered_predicted_labels, labels=misclassified_label_indices)

            # Optional: Normalize the refined CM as well
            if normalize_cm:
                cm_refined = cm_refined.astype('float') / cm_refined.sum(axis=1)[:, np.newaxis]
                cm_refined = np.nan_to_num(cm_refined)

            plt.figure(figsize=(10, 8))
            sns.heatmap(cm_refined, annot=True, fmt='.2f' if normalize_cm else 'g', cmap='Reds',
                        xticklabels=misclassified_class_names,
                        yticklabels=misclassified_class_names)
            plt.title(f'Refined Confusion Matrix (Top 10 Misclassified) for {model_display_name} on {dataset_name}')
            plt.xlabel('Predicted labels')
            plt.ylabel('True labels')
            plt.tight_layout()
            plt.show()
        else:
            print("No samples found for the top misclassified classes in the test set to generate a refined confusion matrix.")
    else:
        print("Could not identify top misclassified classes for refined confusion matrix.")

    return metrics

## Evaluate Head-Only Fine-tuned Model

### Subtask:
Use the `evaluate_and_plot` function to assess the performance of the head-only fine-tuned model on the Food101 test set, including accuracy, F1-score, and a confusion matrix. Also, print the number of trainable parameters for this stage.

In [ ]:
print("Trainable parameters for Head-Only Fine-tuning:")
count_parameters(model)

# Evaluate the head-only fine-tuned model on Food101 test set
head_only_metrics_food101 = evaluate_and_plot(
    model=model,
    trainer=trainer,
    dataset=food["test"],
    id2label_mapping=id2label,
    dataset_name="Food101 Test Set (Head-Only)",
    num_labels=len(id2label),
    model_display_name="ViT Head-Only Fine-tune",
    training_time=vit_head_only_train_time
)

## Evaluate Full Model Fine-tuned Model

### Subtask:
Use the `evaluate_and_plot` function to assess the performance of the full model fine-tuned model on the Food101 test set, including accuracy, F1-score, and a confusion matrix. Also, print the number of trainable parameters for this stage.

In [ ]:
print("Trainable parameters for Full Model Fine-tuning:")
count_parameters(model_full_finetune)

# Evaluate the full model fine-tuned model on Food101 test set
full_finetune_metrics_food101 = evaluate_and_plot(
    model=model_full_finetune,
    trainer=trainer_full,
    dataset=food["test"],
    id2label_mapping=id2label,
    dataset_name="Food101 Test Set (Full Fine-tune)",
    num_labels=len(id2label),
    model_display_name="ViT Full Fine-tune",
    training_time=vit_full_train_time
)

## Evaluate LoRA Fine-tuned Model

### Subtask:
Use the `evaluate_and_plot` function to assess the performance of the LoRA fine-tuned model on the Food101 test set, including accuracy, F1-score, and a confusion matrix. Also, print the number of trainable parameters for this stage.

In [ ]:
print("Trainable parameters for LoRA Fine-tuning:")
model_lora.print_trainable_parameters() # LoRA models have their own method for this

# Evaluate the LoRA fine-tuned model on Food101 test set
lora_metrics_food101 = evaluate_and_plot(
    model=model_lora,
    trainer=trainer_lora,
    dataset=food["test"],
    id2label_mapping=id2label,
    dataset_name="Food101 Test Set (LoRA)",
    num_labels=len(id2label),
    model_display_name="ViT LoRA Fine-tune",
    training_time=vit_lora_train_time
)

## Fine-tuning Last 3 Encoder Blocks + Head

This section will fine-tune the classification head along with the last 3 encoder blocks of the ViT backbone on the Food101 dataset.

In [ ]:
# Re-initialize the model to ensure all layers are frozen initially
model_last_3_blocks = AutoModelForImageClassification.from_pretrained(
    checkpoint,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id,
)

# Freeze all layers first
for param in model_last_3_blocks.parameters():
    param.requires_grad = False

# Unfreeze the classification head
for param in model_last_3_blocks.classifier.parameters():
    param.requires_grad = True

# Unfreeze the last 3 encoder layers
num_encoder_layers = len(model_last_3_blocks.vit.encoder.layer)
for i in range(num_encoder_layers - 3, num_encoder_layers):
    for param in model_last_3_blocks.vit.encoder.layer[i].parameters():
        param.requires_grad = True

print("Model re-initialized. Last 3 encoder blocks and head are unfrozen for fine-tuning.")
count_parameters(model_last_3_blocks)

### Train the Model (Last 3 Blocks + Head)

In [ ]:
training_args_last_3_blocks = TrainingArguments(
    output_dir="/content/drive/MyDrive/ViTExperiments/my_awesome_food_model_last_3_blocks", # Saving to Google Drive
    remove_unused_columns=False,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-4,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=4,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    warmup_steps=0.1,
    logging_steps=10,
    run_name="food101_last_3_blocks",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    push_to_hub=False,
)

trainer_last_3_blocks = Trainer(
    model=model_last_3_blocks,
    args=training_args_last_3_blocks,
    data_collator=data_collator,
    train_dataset=food["train"],
    eval_dataset=food["test"],
    processing_class=image_processor,
    compute_metrics=compute_metrics,
)

print("Starting fine-tuning for last 3 blocks + head...")
# Use the conditional training function
train_result_last_3_blocks = trainer_last_3_blocks.train()
vit_last_3_blocks_train_time = conditional_train_model(model_last_3_blocks, trainer_last_3_blocks, training_args_last_3_blocks, "ViT Last 3 Blocks + Head Fine-tune")
print("Fine-tuning for last 3 blocks + head complete.")

### Evaluate Model (Last 3 Blocks + Head)

In [ ]:
print("Trainable parameters for Last 3 Blocks + Head Fine-tuning:")
count_parameters(model_last_3_blocks)

last_3_blocks_metrics_food101 = evaluate_and_plot(
    model=model_last_3_blocks,
    trainer=trainer_last_3_blocks,
    dataset=food["test"],
    id2label_mapping=id2label,
    dataset_name="Food101 Test Set (Last 3 Blocks + Head)",
    num_labels=len(id2label),
    model_display_name="ViT Last 3 Blocks + Head Fine-tune",
    training_time=vit_last_3_blocks_train_time
)

## Fine-tuning Last 5 Encoder Blocks + Head

This section will fine-tune the classification head along with the last 5 encoder blocks of the ViT backbone on the Food101 dataset.

In [ ]:
# Re-initialize the model to ensure all layers are frozen initially
model_last_5_blocks = AutoModelForImageClassification.from_pretrained(
    checkpoint,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id,
)

# Freeze all layers first
for param in model_last_5_blocks.parameters():
    param.requires_grad = False

# Unfreeze the classification head
for param in model_last_5_blocks.classifier.parameters():
    param.requires_grad = True

# Unfreeze the last 5 encoder layers
num_encoder_layers = len(model_last_5_blocks.vit.encoder.layer)
for i in range(num_encoder_layers - 5, num_encoder_layers):
    for param in model_last_5_blocks.vit.encoder.layer[i].parameters():
        param.requires_grad = True

print("Model re-initialized. Last 5 encoder blocks and head are unfrozen for fine-tuning.")
count_parameters(model_last_5_blocks)

### Train the Model (Last 5 Blocks + Head)

In [ ]:
training_args_last_5_blocks = TrainingArguments(
    output_dir="/content/drive/MyDrive/ViTExperiments/my_awesome_food_model_last_5_blocks", # Saving to Google Drive
    remove_unused_columns=False,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-4,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=4,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    warmup_steps=0.1,
    logging_steps=10,
    run_name="food101_last_5_blocks",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    push_to_hub=False,
)

trainer_last_5_blocks = Trainer(
    model=model_last_5_blocks,
    args=training_args_last_5_blocks,
    data_collator=data_collator,
    train_dataset=food["train"],
    eval_dataset=food["test"],
    processing_class=image_processor,
    compute_metrics=compute_metrics,
)

print("Starting fine-tuning for last 5 blocks + head...")
# Use the conditional training function
vit_last_5_blocks_train_time = conditional_train_model(model_last_5_blocks, trainer_last_5_blocks, training_args_last_5_blocks, "ViT Last 5 Blocks + Head Fine-tune")
print("Fine-tuning for last 5 blocks + head complete.")

### Evaluate Model (Last 5 Blocks + Head)

In [ ]:
print("Trainable parameters for Last 5 Blocks + Head Fine-tuning:")
count_parameters(model_last_5_blocks)

last_5_blocks_metrics_food101 = evaluate_and_plot(
    model=model_last_5_blocks,
    trainer=trainer_last_5_blocks,
    dataset=food["test"],
    id2label_mapping=id2label,
    dataset_name="Food101 Test Set (Last 5 Blocks + Head)",
    num_labels=len(id2label),
    model_display_name="ViT Last 5 Blocks + Head Fine-tune",
    training_time=vit_last_5_blocks_train_time
)

## Summary of Fine-tuning Experiment Results on Food101

In [ ]:
import pandas as pd

results_data = [
    {
        "Strategy": "ViT Head-Only Fine-tune",
        "Trainable Parameters (millions)": round(sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6, 2),
        "Number of Epochs": training_args.num_train_epochs,
        "Learning Rate": training_args.learning_rate,
        "Training Time (seconds)": vit_head_only_train_time,
        "Accuracy": head_only_metrics_food101['accuracy'],
        "F1-score (weighted)": head_only_metrics_food101['f1'],
        "Top-5 Accuracy": head_only_metrics_food101['top_5_accuracy'],
    },
    {
        "Strategy": "ViT Full Fine-tune",
        "Trainable Parameters (millions)": round(sum(p.numel() for p in model_full_finetune.parameters() if p.requires_grad) / 1e6, 2),
        "Number of Epochs": training_args_full.num_train_epochs,
        "Learning Rate": training_args_full.learning_rate,
        "Training Time (seconds)": vit_full_train_time,
        "Accuracy": full_finetune_metrics_food101['accuracy'],
        "F1-score (weighted)": full_finetune_metrics_food101['f1'],
        "Top-5 Accuracy": full_finetune_metrics_food101['top_5_accuracy'],
    },
    {
        "Strategy": "ViT LoRA Fine-tune",
        "Trainable Parameters (millions)": round(sum(p.numel() for p in model_lora.parameters() if p.requires_grad) / 1e6, 2),
        "Number of Epochs": training_args_lora.num_train_epochs,
        "Learning Rate": training_args_lora.learning_rate,
        "Training Time (seconds)": vit_lora_train_time,
        "Accuracy": lora_metrics_food101['accuracy'],
        "F1-score (weighted)": lora_metrics_food101['f1'],
        "Top-5 Accuracy": lora_metrics_food101['top_5_accuracy'],
    },
    {
        "Strategy": "ViT Last 3 Blocks + Head Fine-tune",
        "Trainable Parameters (millions)": round(sum(p.numel() for p in model_last_3_blocks.parameters() if p.requires_grad) / 1e6, 2),
        "Number of Epochs": training_args_last_3_blocks.num_train_epochs,
        "Learning Rate": training_args_last_3_blocks.learning_rate,
        "Training Time (seconds)": vit_last_3_blocks_train_time,
        "Accuracy": last_3_blocks_metrics_food101['accuracy'],
        "F1-score (weighted)": last_3_blocks_metrics_food101['f1'],
        "Top-5 Accuracy": last_3_blocks_metrics_food101['top_5_accuracy'],
    },
    {
        "Strategy": "ViT Last 5 Blocks + Head Fine-tune",
        "Trainable Parameters (millions)": round(sum(p.numel() for p in model_last_5_blocks.parameters() if p.requires_grad) / 1e6, 2),
        "Number of Epochs": training_args_last_5_blocks.num_train_epochs,
        "Learning Rate": training_args_last_5_blocks.learning_rate,
        "Training Time (seconds)": vit_last_5_blocks_train_time,
        "Accuracy": last_5_blocks_metrics_food101['accuracy'],
        "F1-score (weighted)": last_5_blocks_metrics_food101['f1'],
        "Top-5 Accuracy": last_5_blocks_metrics_food101['top_5_accuracy'],
    },
]

results_df = pd.DataFrame(results_data)
results_df = results_df.sort_values(by="Accuracy", ascending=False).reset_index(drop=True)
display(results_df)

## ResNet-50 Full Model Fine-tuning (using `transformers`)

This section will fine-tune a ResNet-50 model using the `AutoImageProcessor` and `AutoModelForImageClassification` from Hugging Face `transformers`, ensuring appropriate preprocessing and consistent training parameters.

In [ ]:
from transformers import AutoImageProcessor, AutoModelForImageClassification
from torchvision.transforms import (CenterCrop,
                                    Compose,
                                    Normalize,
                                    RandomResizedCrop,
                                    ToTensor)

# --- ResNet-50 Specific Preprocessing ---
# Define a model ID for ResNet-50 that is available on Hugging Face Hub
resnet_model_id_hf = "microsoft/resnet-50"

# Load the image processor for ResNet-50
resnet_image_processor = AutoImageProcessor.from_pretrained(resnet_model_id_hf)

# Get the image size from the processor config
resnet_image_size = (
    resnet_image_processor.size["shortest_edge"]
    if "shortest_edge" in resnet_image_processor.size
    else (resnet_image_processor.size["height"], resnet_image_processor.size["width"])
)

# Define the preprocessing transforms for ResNet-50
resnet_transforms_val = Compose([
    CenterCrop(resnet_image_size),
    ToTensor(),
    Normalize(mean=resnet_image_processor.image_mean, std=resnet_image_processor.image_std),
])

resnet_transforms_train = Compose([
    RandomResizedCrop(resnet_image_size),
    ToTensor(),
    Normalize(mean=resnet_image_processor.image_mean, std=resnet_image_processor.image_std),
])

def resnet_transforms_function(examples):
    # Apply appropriate transforms for training vs. evaluation
    if "image" in examples:
        # For training, use random augmentation
        examples["pixel_values"] = [resnet_transforms_train(img.convert("RGB")) for img in examples["image"]]
    return examples

def resnet_val_transforms_function(examples):
    if "image" in examples:
        # For evaluation, use only resizing and normalization
        examples["pixel_values"] = [resnet_transforms_val(img.convert("RGB")) for img in examples["image"]]
    return examples

# Create a new dataset split for ResNet-50 with its specific transforms
# Assuming 'food_full_train' is available from earlier loading
food_resnet = food_full_train.shuffle(seed=42).select(range(5000))
food_resnet = food_resnet.train_test_split(test_size=0.2, shuffle=True, seed=42)
food_resnet["train"] = food_resnet["train"].with_transform(resnet_transforms_function)
food_resnet["test"] = food_resnet["test"].with_transform(resnet_val_transforms_function)

print("ResNet-50 specific image processor and transforms defined and applied to new dataset split.")

The preprocessing steps implemented in the previous cell for the ResNet-50 model are indeed aligned with `transformers` recommendations.

*   An `AutoImageProcessor` specific to `microsoft/resnet-50` is loaded.
*   This processor's configuration (e.g., image size, mean, standard deviation) is used to create `torchvision.transforms.Compose` pipelines.
*   For the training set, `RandomResizedCrop`, `ToTensor`, and `Normalize` are applied.
*   For the test set, `CenterCrop`, `ToTensor`, and `Normalize` are applied.

These are the standard and recommended preprocessing steps for fine-tuning image classification models like ResNet-50 using the Hugging Face `transformers` ecosystem.

In [ ]:
# Load the ResNet-50 model from Hugging Face Transformers
resnet_model_hf = AutoModelForImageClassification.from_pretrained(
    resnet_model_id_hf,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id,
)

# Ensure all parameters require gradients for full fine-tuning
for param in resnet_model_hf.parameters():
    param.requires_grad = True

print("ResNet-50 model loaded and configured for full fine-tuning.")
print(resnet_model_hf)

In [ ]:
from transformers import TrainingArguments, Trainer

# Define TrainingArguments for ResNet-50
training_args_resnet_hf = TrainingArguments(
    output_dir="/content/drive/MyDrive/ViTExperiments/my_awesome_food_resnet50_hf_full_finetune", # New output directory
    remove_unused_columns=False,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=training_args_full.learning_rate, # Same LR as ViT full fine-tune
    per_device_train_batch_size=training_args_full.per_device_train_batch_size,
    gradient_accumulation_steps=training_args_full.gradient_accumulation_steps,
    per_device_eval_batch_size=training_args_full.per_device_eval_batch_size,
    num_train_epochs=training_args_full.num_train_epochs, # Same epochs as ViT full fine-tune
    warmup_steps=training_args_full.warmup_steps,
    logging_steps=training_args_full.logging_steps,
    run_name="food101_resnet50_hf_full_finetune",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    push_to_hub=False,
)

# Initialize the Trainer for ResNet-50
trainer_resnet_hf = Trainer(
    model=resnet_model_hf,
    args=training_args_resnet_hf,
    data_collator=data_collator,
    train_dataset=food_resnet["train"],
    eval_dataset=food_resnet["test"],
    # Ensure processing_class is passed for correct evaluation processing
    # Note: the dataset itself has already been transformed, but the Trainer might use this for internal checks
    # For this setup, the dataset transforms handle the pixel_values directly.
    # We can pass the resnet_image_processor here, but the actual transformation for eval is done by food_resnet["test"].with_transform
    processing_class=resnet_image_processor,
    compute_metrics=compute_metrics,
)

print("Starting ResNet-50 (Hugging Face) full model fine-tuning...")
# Use the conditional training function
resnet_hf_full_train_time = conditional_train_model(resnet_model_hf, trainer_resnet_hf, training_args_resnet_hf, "ResNet-50 HF Full Fine-tune")
print("ResNet-50 (Hugging Face) full model fine-tuning complete.")

### Evaluate ResNet-50 (Hugging Face) Full Fine-tuned Model

In [ ]:
print("Trainable parameters for ResNet-50 (Hugging Face) Full Model Fine-tuning:")
count_parameters(resnet_model_hf)

resnet_hf_full_metrics_food101 = evaluate_and_plot(
    model=resnet_model_hf,
    trainer=trainer_resnet_hf,
    dataset=food_resnet["test"],
    id2label_mapping=id2label,
    dataset_name="Food101 Test Set (ResNet-50 HF Full Fine-tune)",
    num_labels=len(id2label),
    model_display_name="ResNet-50 HF Full Fine-tune",
    training_time=resnet_hf_full_train_time
)

In [ ]:
results_data.append(
    {
        "Strategy": "ResNet-50 HF Full Fine-tune",
        "Trainable Parameters (millions)": round(sum(p.numel() for p in resnet_model_hf.parameters() if p.requires_grad) / 1e6, 2),
        "Number of Epochs": training_args_resnet_hf.num_train_epochs,
        "Learning Rate": training_args_resnet_hf.learning_rate,
        "Training Time (seconds)": resnet_hf_full_train_time,
        "Accuracy": resnet_hf_full_metrics_food101['accuracy'],
        "F1-score (weighted)": resnet_hf_full_metrics_food101['f1'],
        "Top-5 Accuracy": resnet_hf_full_metrics_food101['top_5_accuracy'],
    }
)

# Re-create and display the DataFrame with the new results
results_df = pd.DataFrame(results_data)
results_df = results_df.sort_values(by="Accuracy", ascending=False).reset_index(drop=True)
display(results_df)